# 6. Question 2: is the race signal explained by cardiac structure?

Structural diagnoses differ in frequency between the groups (Table 1), so the race probes might simply detect structure. Four analyses test this (Black versus White):

0. **Structure only**: a logistic model on the structural variables alone (age, sex, LVEF, septal and posterior (EchoNext) or inferolateral (MIMIC-IV) wall thickness, and PASP (EchoNext) or tricuspid regurgitation pressure gradient (MIMIC-IV) with a missing indicator), fitted with the same cross-fitting.
1. **No recorded SHD**: the race probes refitted on ECGs negative for all 11 components (EchoNext: composite SHD = 0; MIMIC-IV: `shd_full` = 0).
2. **Matching**: each Black patient paired with a White patient of the same sex and clinical setting by optimal one-to-one assignment on age, LVEF, the two wall thicknesses and calendar year (notebook 00, section 0.9); the score is the probe of notebook 05, and the result is the within-pair concordance.
3. **Removal of structural components**: every embedding dimension is regressed on the structural variables in the fitting rows, and the probes are refitted on the residuals, which keep only what structure cannot explain linearly; compared with a probe on the same rows without removal.

Analyses that need the structural variables use ECGs with LVEF and both wall thicknesses recorded ("core" rows). In MIMIC-IV, the clinical setting comes from the ECG time stamp and the calendar year from the three-year anchor group.

**Paper items:** Fig. 1c, Supplementary Table 4, Table 2 (question 2).

Run time: about 8 minutes for both cohorts on one RTX 5090.

In [ ]:
%run ./00_settings.ipynb
OUT = results("06_cardiac_structure")


def oof_scores(cohort, name):
    z = np.load(WORK_DIR / cohort / "race_probes" / f"oof_{name}.npz")
    return z["p4"].astype(float), z["sbw"].astype(float)


def removal_transform(Sv):
    """Method 3: regress every embedding dimension on the standardised structural variables of the fitting rows (last column may be missing:
    median of the fitting rows plus a missing indicator) and keep the residuals of the fitting, validation and evaluation rows."""
    def transform(k, Xtr, Xva, Xev, tr, va, ev):
        med = np.nanmedian(Sv[tr, 5])

        def zmat(r):
            z = Sv[r].copy()
            miss = np.isnan(z[:, 5])
            z[miss, 5] = med
            return np.column_stack([z, miss.astype(float)])
        Zt, Zv, Ze = zmat(tr), zmat(va), zmat(ev)
        mu, sd = Zt.mean(0), Zt.std(0)
        sd[sd == 0] = 1
        Zt, Zv, Ze = ((z - mu) / sd for z in (Zt, Zv, Ze))
        aug = lambda z: to_dev(np.column_stack([np.ones(len(z)), z]))
        At, Av, Ae = aug(Zt), aug(Zv), aug(Ze)
        coef = torch.linalg.pinv(At) @ Xtr
        v0 = col_mean_var(Xtr)[1]
        for X_, A_ in ((Xtr, At), (Xva, Av), (Xev, Ae)):           # residuals in place, in row chunks
            for i in range(0, X_.shape[0], 20000):
                X_[i:i + 20000] -= A_[i:i + 20000] @ coef
        r2 = float((1 - col_mean_var(Xtr)[1] / torch.where(v0 == 0, torch.ones_like(v0), v0)).mean())
        return Xtr, Xva, Xev, {"r2_structure_mean": r2}
    return transform


def run_cohort(cohort, D, g_row, g_lat, Sv, core, neg, Dt, emb):
    """The four analyses for one cohort. Sv: structural variables of every ECG (age, male, LVEF, wall 1, wall 2, PASP or TR gradient)."""
    lat = D["lat_idx"]
    core_lat = core[lat]
    summary = {"core_fraction_latest": float(core_lat[g_lat >= 0].mean()), "n_core_black": int((core_lat & (g_lat == B)).sum()),
               "n_core_white": int((core_lat & (g_lat == W)).sum())}
    # ---- 0: structure only
    P4, struct_sb, _ = probe_folds(Sv, D, g_row, row_mask=core, do_impute=True)
    rec, _ = evaluate(P4, struct_sb, g_lat, SEED_STRUCTURE, subset=core_lat)
    m0 = pd.DataFrame([{"cohort": cohort, "model": "structure only", **rec}])
    # ---- 2: matching (scores of notebook 05, no refit)
    scores = {fm: oof_scores(cohort.lower().replace("-iv", ""), fm)[1] for fm in FMS}
    scores["structure only"] = struct_sb
    pp, bal, info = match(Dt, B, W)
    conc = concordance(pp, scores).assign(cohort=cohort)
    summary["matching"] = info
    # ---- 1: no recorded SHD
    rowsA = []
    for fm in FMS:
        P4, SB, _ = probe_folds(emb(fm), D, g_row, row_mask=neg)
        rec, _ = evaluate(P4, SB, g_lat, SEED_STRUCTURE)
        rowsA.append({"cohort": cohort, "model": fm, **rec})
    # ---- 3: removal of structural components (Black vs White), against a probe on the same rows
    rowsC = []
    bw_core = np.flatnonzero(core_lat & np.isin(g_lat, [B, W]))
    boots = boot_idx(len(bw_core), SEED_STRUCTURE + 3)
    yb = g_lat[bw_core] == B
    for fm in FMS:
        X = emb(fm)
        _, s_base, _ = probe_folds(X, D, g_row, row_mask=core, tasks=("bw",))
        _, s_rem, info_c = probe_folds(X, D, g_row, row_mask=core, transform=removal_transform(Sv), tasks=("bw",))
        s0, s1 = s_base[bw_core], s_rem[bw_core]
        b0 = np.array([fast_auc(yb[ix], s0[ix]) for ix in boots])
        b1 = np.array([fast_auc(yb[ix], s1[ix]) for ix in boots])
        for variant, s, bs in (("baseline", s0, b0), ("removed", s1, b1)):
            rec = {"cohort": cohort, "model": fm, "variant": variant, "bw_auroc": fast_auc(yb, s), "bw_lo": ci95(bs)[0], "bw_hi": ci95(bs)[1],
                   "n_black": int(yb.sum()), "n_white": int((~yb).sum())}
            if variant == "removed":
                rec.update(delta=rec["bw_auroc"] - fast_auc(yb, s0), delta_lo=ci95(b1 - b0)[0], delta_hi=ci95(b1 - b0)[1],
                           r2_structure_mean=float(np.mean([i["r2_structure_mean"] for i in info_c])))
            rowsC.append(rec)
    key = cohort.lower().replace("-iv", "")
    m0.to_csv(OUT / f"{key}_structure_only.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(rowsA).to_csv(OUT / f"{key}_no_recorded_shd.csv", index=False, encoding="utf-8-sig")
    conc.to_csv(OUT / f"{key}_matched_concordance.csv", index=False, encoding="utf-8-sig")
    bal.to_csv(OUT / f"{key}_matched_balance.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(rowsC).to_csv(OUT / f"{key}_structure_removed.csv", index=False, encoding="utf-8-sig")
    save_json(summary, OUT / f"{key}_summary.json")
    return m0, pd.DataFrame(rowsA), conc, bal, pd.DataFrame(rowsC), summary

## 6.1 EchoNext

In [ ]:
D = load_design("echonext")
D["lat_idx"] = D["lat_row"]
A_rows = echonext_rows()
num = lambda c: pd.to_numeric(A_rows[c], errors="coerce").to_numpy(float)
Sv = np.column_stack([num("age_at_ecg"), (A_rows.sex.astype(str).str.lower() == "male").to_numpy(float), num("lvef_value"), num("ivs_measurement"),
                      num("lvpw_measurement"), num("pasp_value")])
core = np.isfinite(Sv[:, 2]) & np.isfinite(Sv[:, 3]) & np.isfinite(Sv[:, 4])
neg = (A_rows[E_COMP] == 0).to_numpy()
lat = D["lat_row"]
g_lat = D["g"]
Dt = {"keep": g_lat >= 0, "core": core[lat], "y4": g_lat.astype(float), "age": Sv[lat, 0], "lvef": Sv[lat, 2], "ivs": Sv[lat, 3], "lvpw": Sv[lat, 4],
      "year": num("acquisition_year")[lat], "male": Sv[lat, 1], "loc": A_rows.location_setting.astype(str).to_numpy()[lat]}
out_e = run_cohort("EchoNext", D, D["g"][D["pi_of_row"]], g_lat, Sv, core, neg, Dt, emb_echonext)
print("matching:", out_e[5]["matching"])
pd.concat([out_e[0][["model", "bw_auroc", "bw_lo", "bw_hi"]], out_e[1][["model", "bw_auroc", "bw_lo", "bw_hi"]]]).round(3)

In [ ]:
display(out_e[2].round(3), out_e[4].round(3))

## 6.2 MIMIC-IV

In [ ]:
C = load_mimic_cohort()
Dm = load_design("mimic")
lat_m = C.loc[Dm["lat_idx"]].reset_index(drop=True)
g_row = race_index(C.race6)
g_lat = g_row[Dm["lat_idx"]]
Sv = C[["age", "male", "ef", "sept", "infl", "tr_grad"]].to_numpy(float)
core = (C.ef.notna() & C.sept.notna() & C.infl.notna()).to_numpy()
neg = (C.shd_full == 0).to_numpy()
Dt = {"keep": g_lat >= 0, "core": core[Dm["lat_idx"]], "y4": g_lat.astype(float), "age": lat_m.age.to_numpy(float), "lvef": lat_m.ef.to_numpy(float),
      "ivs": lat_m.sept.to_numpy(float), "lvpw": lat_m.infl.to_numpy(float), "year": lat_m.year_approx.to_numpy(float), "male": lat_m.male.to_numpy(float),
      "loc": lat_m.setting.to_numpy()}
rows_m = C.row.to_numpy()
out_m = run_cohort("MIMIC-IV", Dm, g_row, g_lat, Sv, core, neg, Dt, lambda name: emb_mimic(name)[rows_m])
print("matching:", out_m[5]["matching"])
pd.concat([out_m[0][["model", "bw_auroc", "bw_lo", "bw_hi"]], out_m[1][["model", "bw_auroc", "bw_lo", "bw_hi"]]]).round(3)

In [ ]:
display(out_m[2].round(3), out_m[4].round(3))